# Phase 1: Data Understanding

The initial step is to load the dataset and investigate its structure. We want to understand the features, inspect missing values, and obtain initial descriptive statistics. 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display settings
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

df = pd.read_csv('Wholesale customers data.csv')
display(df.head())

### Basic Information
Let's check the data types and if there are any null values.

In [ ]:
df.info()

In [ ]:
print("Missing values in each column:")
display(df.isnull().sum())

### Descriptive Statistics
Here we observe the distributions explicitly. Notice the substantial differences between median (50%) and the maximum value across continuous categories.

In [ ]:
display(df.describe())

### Visualizing Data Disruptions (Skewness & Outliers)
While earlier we used `df.describe()` to mathematically identify variations, providing a visual representation paints a much clearer picture for clients. Below, we've produced two key visualization types:
1. **Histograms (with Mean and Median lines)**: This shows the distribution shape. We can clearly see the distribution is heavily pulled to the right, signifying 'Positive Skewness'. The 'Mean' is heavily disrupted upwards by massive spenders, abandoning the 'Median'. 
2. **Boxplots**: A standard approach to find outliers. The solid boxes represent where the central 50% of our clients lie. The dots scattered high above are our 'disruptions'—individual clients purchasing massively larger quantities than a normal customer.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

continuous_features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']

# 1. Histograms to show Skewness
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
fig.suptitle('Distributions of Annual Spending (Highlighting Extreme Skewness)', fontsize=18, y=1.02)
axes = axes.flatten()

for i, col in enumerate(continuous_features):
    sns.histplot(df[col], kde=True, ax=axes[i], color='skyblue', bins=40)
    median_val = df[col].median()
    mean_val = df[col].mean()
    axes[i].axvline(median_val, color='red', linestyle='--', linewidth=2, label=f'Median: {median_val:.0f}')
    axes[i].axvline(mean_val, color='navy', linestyle='-', linewidth=2, label=f'Mean: {mean_val:.0f}')
    axes[i].set_title(f'{col} Spending')
    axes[i].legend()

plt.tight_layout()
plt.show()

# 2. Boxplots to emphasize Outliers
plt.figure(figsize=(14, 7))
sns.boxplot(data=df[continuous_features], palette='pastel')
plt.title('Boxplots of Annual Spending (Highlighting Extreme Outliers vs Interquartile Range)', fontsize=16)
plt.ylabel('Annual Spending (Monetary Units)')
plt.xticks(rotation=45)
plt.show()

# Phase 2: Data Cleaning & Preprocessing
Our goals in this phase are to eliminate any basic flaws (like full-row duplicates), and aggressively tackle the extreme positive skew present in our spending data via Log Transformation and Standard Scaling. We will then save the cleaned result to a dedicated `data/` folder.

In [ ]:

print(f"Original shape: {df.shape}")
df = df.drop_duplicates()
print(f"Shape after duplicates removed: {df.shape}")

from sklearn.preprocessing import StandardScaler
import numpy as np
import os

spend_features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']

# Log Transform (np.log is unsafe with 0, np.log1p handles 0 by doing log(1+x))
log_data = np.log1p(df[spend_features])

# Standard Scaling ensures Euclidean distances aren't dominated by raw magnitude
scaler = StandardScaler()
scaled_data = scaler.fit_transform(log_data)
df_scaled = pd.DataFrame(scaled_data, columns=spend_features, index=df.index)

# Re-attach Channel & Region
df_cleaned = df[['Channel', 'Region']].join(df_scaled)

# Ensure 'data' directory exists and save
os.makedirs('data', exist_ok=True)
df_cleaned.to_csv('data/Cleaned_Wholesale_Data.csv', index=False)
print("Cleaned & Scaled data saved to 'data/Cleaned_Wholesale_Data.csv'!")
display(df_cleaned.head())


# Phase 3: Exploratory Data Analysis
We will visualize the transformed distributions to ensure they are more normalized, and map out the correlations between different categorical spends.

In [ ]:

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_cleaned[spend_features], palette='muted')
plt.title("Boxplot of Spend Features (Post Log-Transform & Scaling)", fontsize=14)
plt.xticks(rotation=45)
plt.show()


In [ ]:

plt.figure(figsize=(8, 6))
sns.heatmap(df_cleaned[spend_features].corr(), annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Matrix of Product Categories", fontsize=14)
plt.show()


# Phase 4: Clustering & Customer Segmentation
Using K-Means clustering to dynamically segment customers based on their spending behavior. We will use the Elbow Method and Silhouette Score to find the mathematical 'best' number of clusters ($K$).

In [ ]:

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore') # ignore K-means warning

inertia = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(df_scaled)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(df_scaled, clusters))

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(K_range, inertia, 'bo-', label='Inertia (Within-Cluster Sum of Squares)')
ax1.set_xlabel('Number of Clusters (k)', fontsize=12)
ax1.set_ylabel('Inertia', color='b', fontsize=12)
ax1.tick_params(axis='y', labelcolor='b')

ax2 = ax1.twinx()
ax2.plot(K_range, silhouette_scores, 'ro-', label='Silhouette Score')
ax2.set_ylabel('Silhouette Score', color='r', fontsize=12)
ax2.tick_params(axis='y', labelcolor='r')

plt.title('Determining Optimal K: Elbow Method & Silhouette Score', fontsize=14)
plt.grid(False)
plt.show()


Based on the Silhouette score hitting a peak and the inertia showing an 'elbow', `k=3` or `k=5` are great candidates. We will proceed with **K=3** for interpretability.

In [ ]:

optimal_k = 3
final_kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['Cluster'] = final_kmeans.fit_predict(df_scaled)

print("--- CLUSTER CHARACTERISTICS ---")
target_insight = df.groupby('Cluster')[spend_features].mean().round(2)
display(target_insight)


In [ ]:

# PCA Visual validation
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
reduced_data = pca.fit_transform(df_scaled)

plt.figure(figsize=(10, 7))
sns.scatterplot(x=reduced_data[:,0], y=reduced_data[:,1], hue=df['Cluster'], palette='Set1', s=100, alpha=0.8)
plt.title('2D PCA Projection of the Clusters', fontsize=14)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend(title='Cluster')
plt.show()
